- frame_dir (str): The identifier of the corresponding video. (name of file)
- total_frames (int): The number of frames in this video. (len of 'keypoints')
- img_shape (tuple[int]): The shape of a video frame, a tuple with two elements, in the format of (height, width). Only required for 2D skeletons. (got it)
- original_shape (tuple[int]): Same as img_shape. (got it)
- label (int): The action label. ('overhead press')
- keypoint (np.ndarray, with shape [M x T x V x C]): The keypoint annotation. M: number of persons; T: number of frames (same as total_frames); V: number of keypoints (25 for NTURGB+D 3D skeleton, 17 for CoCo, 18 for OpenPose, etc. ); C: number of dimensions for keypoint coordinates (C=2 for 2D keypoint)
- keypoint_score (np.ndarray, with shape [M x T x V]): The confidence score of keypoints. Only required for 2D skeletons.


# Load JSONs


In [118]:
# %pip install pandas

import pandas as pd
import json
import pickle
import os

In [119]:
# Settings
base_dir = '../../data'
# sample_class = 'correct'  # 'knees_error', 'elbows_error'
# sample_class = 'knees_error' #'correct'  'knees_error', 'elbows_error'
sample_class = 'elbows_error'  # 'knees_error', 'elbows_error'

extract_main_person = False

In [120]:
# Path to the folder with JSON files
json_folder = os.path.join(base_dir, 'ohp_poses_corrected', sample_class)


# Dictionary to store all loaded JSON data
all_data = {}

# Loop through all .json files in the folder
for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        filepath = os.path.join(json_folder, filename)
        with open(filepath, 'r') as file:
            try:
                data = json.load(file)
                key = os.path.splitext(filename)[0]  # filename without .json
                all_data[key] = data
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse {filename}, skipping.")

# Example: print one loaded entry
print(all_data.keys())

dict_keys(['62805_6', '62866_3', '62868_4', '62876_2', '62947_2', '62989_7', '62992_5', '62993_1', '63009_4', '63028_3', '63041_9', '63094_1', '63097_1', '63106_6', '63159_8', '63164_2', '63203_3', '63207_1', '63241_1', '63248_1', '63296_2', '63309_8', '63313_2', '63364_1', '63378_5', '63390_7', '63422_3', '63425_1', '63427_4', '63451_3', '63474_2', '63526_1', '63672_1', '63706_7', '63725_6', '63729_5', '63766_9', '63783_4', '63784_2', '63815_1', '63886_1', '63910_2', '63915_6', '63918_2', '63919_3', '63992_1', '63995_1', '64000_1', '64071_1', '64079_2', '64099_1', '64108_11', '64120_1', '64142_1', '64164_1', '64194_1', '64198_6', '64232_1', '64271_3', '64307_1', '64336_1', '64349_7', '64361_2', '64386_3', '64420_3', '64450_2', '64460_3', '64482_1', '64503_2', '64546_1', '64591_3', '64607_2', '64624_3', '64673_3', '64702_1', '64708_1', '64710_3', '64775_6', '64798_1', '64799_1', '64800_1', '64804_3', '64893_3', '64899_1', '64908_5', '64922_5', '64935_5', '65009_1', '65010_1', '65025_1'

In [121]:
# print('No. people: ',len(all_data.get('62794_6').get('keypoints')[0].keys()))

all_data


In [122]:
# print('No. frames: ', len(all_data.get('62794_6').get('keypoints')))

In [123]:
# %pip install opencv-python

# Get the information of the VIDEOS


In [124]:
import cv2

# Path to the folder containing .mp4 videos
video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# Dictionary to hold video metadata
video_info = {}

# Loop through all files in the folder
for filename in os.listdir(video_folder):
    if filename.lower().endswith('.mp4'):
        video_path = os.path.join(video_folder, filename)
        video_name = os.path.splitext(filename)[0]

        # Open video file
        vid = cv2.VideoCapture(video_path)

        if not vid.isOpened():
            print(f"❌ Failed to open: {filename}")
            continue

        # Get properties
        width = vid.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = vid.get(cv2.CAP_PROP_FRAME_HEIGHT)
        fps = vid.get(cv2.CAP_PROP_FPS)
        frame_count = vid.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = frame_count / fps if fps else 0

        # Store in dictionary
        video_info[video_name] = {
            "width": int(width),
            "height": int(height),
            "fps": round(fps, 2),
            "frame_count": int(frame_count),
            "duration_sec": round(duration, 2)
        }

        vid.release()

# Print or save the results
output_path = os.path.join(video_folder, 'video_properties.json')
with open(output_path, 'w') as f:
    json.dump(video_info, f, indent=2)

print(f"✅ Processed {len(video_info)} videos. Info saved to: {output_path}")

✅ Processed 459 videos. Info saved to: ../../data\ohp_labeled\elbows_error\video_properties.json


In [125]:
with open(os.path.join(video_folder, 'video_properties.json'), 'r') as file:
    video_properties = json.load(file)

In [126]:
video_properties

{'62805_6': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 48,
  'duration_sec': 1.6},
 '62866_3': {'width': 480,
  'height': 270,
  'fps': 30.0,
  'frame_count': 77,
  'duration_sec': 2.57},
 '62868_4': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 73,
  'duration_sec': 2.43},
 '62876_2': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 82,
  'duration_sec': 2.73},
 '62947_2': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 116,
  'duration_sec': 3.87},
 '62989_7': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 85,
  'duration_sec': 2.83},
 '62992_5': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 160,
  'duration_sec': 5.33},
 '62993_1': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 320,
  'duration_sec': 10.67},
 '63009_4': {'width': 480,
  'height': 270,
  'fps': 30.0,
  'frame_count': 62,
  'duration_sec': 2.07},
 '63028_3': {'width': 480,
  'height': 270,
  'fps':

# Detection of people in the videos


In [127]:
# this detects the number of people in each video, in the frame keypoints[1] (the second person)

people_lst = []
for i in all_data.keys():
    print('No. people: ',len(all_data.get(i).get('keypoints')[1].keys()))
    people_lst.append([i, len(all_data.get(i).get('keypoints')[1].keys())])

No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  2
No. people:  3
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  3
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  4
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people

In [128]:
# videos with no people in the second frame
for i in people_lst:
    if i[1]==0:
        print(i)




['68283_3', 0]
['69116_4', 0]


In [129]:
# How many people are in each video at most?

people_lst = []
for i in all_data.keys():
    people_visible = []
    for j in range(len(all_data.get(i).get('keypoints'))):
        if len(all_data.get(i).get('keypoints')[j])>0:
            people_visible.append(len(all_data.get(i).get('keypoints')[j].keys()))
    people_lst.append([i, max(people_visible)])
people_lst


[['62805_6', 1],
 ['62866_3', 1],
 ['62868_4', 1],
 ['62876_2', 1],
 ['62947_2', 2],
 ['62989_7', 5],
 ['62992_5', 2],
 ['62993_1', 3],
 ['63009_4', 4],
 ['63028_3', 4],
 ['63041_9', 1],
 ['63094_1', 2],
 ['63097_1', 3],
 ['63106_6', 1],
 ['63159_8', 1],
 ['63164_2', 1],
 ['63203_3', 2],
 ['63207_1', 1],
 ['63241_1', 1],
 ['63248_1', 2],
 ['63296_2', 1],
 ['63309_8', 1],
 ['63313_2', 1],
 ['63364_1', 1],
 ['63378_5', 2],
 ['63390_7', 1],
 ['63422_3', 1],
 ['63425_1', 2],
 ['63427_4', 3],
 ['63451_3', 1],
 ['63474_2', 1],
 ['63526_1', 2],
 ['63672_1', 1],
 ['63706_7', 2],
 ['63725_6', 2],
 ['63729_5', 3],
 ['63766_9', 2],
 ['63783_4', 1],
 ['63784_2', 3],
 ['63815_1', 2],
 ['63886_1', 1],
 ['63910_2', 3],
 ['63915_6', 2],
 ['63918_2', 1],
 ['63919_3', 1],
 ['63992_1', 5],
 ['63995_1', 3],
 ['64000_1', 5],
 ['64071_1', 1],
 ['64079_2', 1],
 ['64099_1', 2],
 ['64108_11', 2],
 ['64120_1', 2],
 ['64142_1', 1],
 ['64164_1', 1],
 ['64194_1', 2],
 ['64198_6', 1],
 ['64232_1', 1],
 ['64271_3', 

In [130]:
# Ressume of the max number of people per video
# Conclusion: is worth it to work with the 2 and 3 people videos.

from collections import Counter

# Get only the max number of people per video
max_people_per_video = [item[1] for item in people_lst]

# Count how many times each number appears
summary = Counter(max_people_per_video)

# Print summary sorted by number of people
for num_people in sorted(summary):
    print(f"{summary[num_people]} videos with {num_people} people at most")


261 videos with 1 people at most
122 videos with 2 people at most
53 videos with 3 people at most
13 videos with 4 people at most
9 videos with 5 people at most
1 videos with 7 people at most


In [131]:
# Get ID_video with max number of people per video, only when is greater than 1

from collections import defaultdict

# Diccionario para agrupar por número de personas máximas por video
videos_by_people_count = defaultdict(list)

# Obtener la cantidad máxima de personas por video
for video_id, video_data in all_data.items():
    max_people = 0
    for frame in video_data['keypoints']:
        people_in_frame = len(frame)
        if people_in_frame > max_people:
            max_people = people_in_frame

    if max_people > 1:
        videos_by_people_count[max_people].append(video_id)

# Mostrar resultados como: id video | N people in screen
for people_count in sorted(videos_by_people_count.keys()):
    for video_id in videos_by_people_count[people_count]:
        print(f"{video_id} | {people_count} people in screen")


62947_2 | 2 people in screen
62992_5 | 2 people in screen
63094_1 | 2 people in screen
63203_3 | 2 people in screen
63248_1 | 2 people in screen
63378_5 | 2 people in screen
63425_1 | 2 people in screen
63526_1 | 2 people in screen
63706_7 | 2 people in screen
63725_6 | 2 people in screen
63766_9 | 2 people in screen
63815_1 | 2 people in screen
63915_6 | 2 people in screen
64099_1 | 2 people in screen
64108_11 | 2 people in screen
64120_1 | 2 people in screen
64194_1 | 2 people in screen
64307_1 | 2 people in screen
64336_1 | 2 people in screen
64349_7 | 2 people in screen
64503_2 | 2 people in screen
64673_3 | 2 people in screen
64702_1 | 2 people in screen
64708_1 | 2 people in screen
64804_3 | 2 people in screen
64899_1 | 2 people in screen
64908_5 | 2 people in screen
65343_1 | 2 people in screen
65348_4 | 2 people in screen
65524_2 | 2 people in screen
65771_2 | 2 people in screen
65783_2 | 2 people in screen
65971_1 | 2 people in screen
66322_3 | 2 people in screen
66492_1 | 2 p

In [132]:
# # Copy the VIDEOS to a new folder to manually check them
# # Conclusion: nothing really crazy happening here, the videos are quite normal. The main person is visible most of the time.


# import os
# import shutil

# # Crear lista de videos con más de una persona
# multi_person_videos = []

# for video_id, video_data in all_data.items():
#     max_people = max(len(frame) for frame in video_data['keypoints'])
#     if max_people > 1:
#         multi_person_videos.append(video_id)

# # Definir carpetas
# destination_base = os.path.join(base_dir, "videos with multiple people")
# destination_class_folder = os.path.join(destination_base, sample_class)
# source_video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# # Crear carpetas si no existen
# os.makedirs(destination_class_folder, exist_ok=True)

# # Copiar los archivos de video
# for video_id in multi_person_videos:
#     source_file = os.path.join(source_video_folder, f"{video_id}.mp4")
#     destination_file = os.path.join(destination_class_folder, f"{video_id}.mp4")

#     if os.path.exists(source_file):
#         shutil.copy2(source_file, destination_file)
#         print(f"✅ Copied: {video_id}.mp4")
#     else:
#         print(f"⚠️ Video not found: {video_id}.mp4")

# print(f"\n🎯 Completed copying {len(multi_person_videos)} videos to {destination_class_folder}")


In [133]:
for i in people_lst:
    if i[1]>3:
        print(i)

['62989_7', 5]
['63009_4', 4]
['63028_3', 4]
['63992_1', 5]
['64000_1', 5]
['64271_3', 4]
['64482_1', 4]
['64799_1', 5]
['64893_3', 5]
['64922_5', 4]
['66242_6', 4]
['66251_9', 5]
['67066_4', 5]
['67451_2', 5]
['68880_3', 4]
['69055_3', 4]
['69254_2', 4]
['69780_6', 5]
['70924_4', 4]
['71960_2', 4]
['74030_6', 4]
['76981_4', 4]
['77575_1', 7]


In [134]:
# # Define boxes for each person in the frame

# def get_bounding_box(keypoints, threshold=0.0):
#     """
#     keypoints: list of (x, y, confidence) or (x, y)
#     """
#     valid_points = []
#     for kp in keypoints:
#         if len(kp) == 3:
#             x, y, conf = kp
#             if conf >= threshold:
#                 valid_points.append((x, y))
#         elif len(kp) == 2:
#             x, y = kp
#             valid_points.append((x, y))

#     if not valid_points:
#         return None

#     xs, ys = zip(*valid_points)
#     return min(xs), min(ys), max(xs), max(ys)

In [135]:
# def bbox_area(bbox):
#     x_min, y_min, x_max, y_max = bbox
#     return (x_max - x_min) * (y_max - y_min)

In [136]:
#all_data.get('80557_5').get('keypoints')

In [137]:
#video_properties.get('80557_5')

In [138]:
len(all_data.keys())

459

# Nacho: Track same people through the different frames


In [139]:
import os
from collections import defaultdict

# Configuration parameters
frame_gap_threshold = 3  # Number of frames that counts as a significant disappearance

# Output folder for the corrected JSONs (not yet used but prepared)
json_output_folder = os.path.join(base_dir, 'ohp_poses_corrected', sample_class)
os.makedirs(json_output_folder, exist_ok=True)

# Dictionary to hold videos that may require ID tracking
candidates_for_tracking = {}

# Only process videos where more than 1 person appears at some point
for video_id, video_data in all_data.items():
    keypoints = video_data['keypoints']
    person_frames = defaultdict(list)  # {person_id: [frame indices where person appears]}

    # Collect which frames each person_id appears in
    for frame_idx, frame_data in enumerate(keypoints):
        for person_id in frame_data.keys():
            person_frames[person_id].append(frame_idx)

    # Look for person IDs that disappear and reappear (i.e., have big gaps)
    fragmented_ids = []
    for pid, frames in person_frames.items():
        if len(frames) < 2:
            continue  # skip IDs that appear only once
        # Calculate the frame-to-frame gaps
        gaps = [b - a for a, b in zip(frames[:-1], frames[1:])]
        max_gap = max(gaps) if gaps else 0
        if max_gap >= frame_gap_threshold:
            fragmented_ids.append((pid, max_gap))

    # If we found any ID with gaps, mark this video for tracking
    if fragmented_ids:
        candidates_for_tracking[video_id] = {
            "fragmented_ids": fragmented_ids,
            "person_frames": dict(person_frames)
        }

print(f"🎯 Detected {len(candidates_for_tracking)} videos with potential ID fragmentation.\n")

# Example output: show first 5 videos with issues
for vid, data in list(candidates_for_tracking.items())[:5]:
    print(f"📹 Video: {vid}")
    for pid, gap in data["fragmented_ids"]:
        print(f"   ⚠️ Person ID {pid} has a gap of {gap} frames")
    print("")


🎯 Detected 82 videos with potential ID fragmentation.

📹 Video: 62947_2
   ⚠️ Person ID 3 has a gap of 7 frames

📹 Video: 62989_7
   ⚠️ Person ID 21 has a gap of 4 frames

📹 Video: 62992_5
   ⚠️ Person ID 38 has a gap of 3 frames
   ⚠️ Person ID 42 has a gap of 3 frames

📹 Video: 62993_1
   ⚠️ Person ID 87 has a gap of 4 frames

📹 Video: 63028_3
   ⚠️ Person ID 62 has a gap of 4 frames



In [140]:
import numpy as np

def pose_similarity(pose1, pose2, conf1=None, conf2=None, conf_threshold=0.3):
    """
    Compare two poses (list of 17 keypoints) using average L2 distance.
    Optionally uses confidence scores to ignore low-confidence keypoints.
    """
    assert len(pose1) == len(pose2), "Poses must have the same number of keypoints"
    
    valid_dists = []
    for i in range(len(pose1)):
        if len(pose1[i]) < 2 or len(pose2[i]) < 2:
            continue
        if conf1 and conf1[i] < conf_threshold:
            continue
        if conf2 and conf2[i] < conf_threshold:
            continue
        
        dist = np.linalg.norm(np.array(pose1[i][:2]) - np.array(pose2[i][:2]))
        valid_dists.append(dist)
    
    if len(valid_dists) == 0:
        return float('inf')  # no reliable points to compare
    return np.mean(valid_dists)


In [141]:
import copy

# Configurable thresholds
pose_dist_threshold = 25.0  # average pixel distance between keypoints
min_matched_keypoints = 5   # minimum number of valid keypoints for comparison

# Folder to store corrected JSONs
corrected_json_path = os.path.join(base_dir, 'ohp_poses_corrected', sample_class)
os.makedirs(corrected_json_path, exist_ok=True)

for video_id, info in candidates_for_tracking.items():
    original_data = all_data[video_id]
    corrected_data = copy.deepcopy(original_data)
    keypoints = corrected_data['keypoints']

    for fragmented_id, _ in info['fragmented_ids']:
        frames_present = info['person_frames'][fragmented_id]
        frames_present.sort()
        gaps = [b - a for a, b in zip(frames_present[:-1], frames_present[1:])]

        for i, gap in enumerate(gaps):
            if gap < frame_gap_threshold:
                continue
            
            # Last frame before disappearance
            frame_before = frames_present[i]
            pose_before = keypoints[frame_before].get(fragmented_id, None)

            if not pose_before:
                continue

            # Look ahead in the gap to find new IDs
            for f in range(frames_present[i] + 1, frames_present[i + 1]):
                frame_data = keypoints[f]
                for candidate_id, candidate_pose in list(frame_data.items()):  # <-- fixed here
                    if candidate_id == fragmented_id:
                        continue

                    # Calculate similarity
                    dist = pose_similarity(pose_before, candidate_pose)

                    if dist < pose_dist_threshold:
                        print(f"🔄 Reassigning ID {candidate_id} → {fragmented_id} in video {video_id} (frame {f})")

                        # Perform reassignment
                        keypoints[f][fragmented_id] = keypoints[f].pop(candidate_id)

                        # Update tracking info
                        info['person_frames'][fragmented_id].append(f)
                        info['person_frames'][candidate_id].remove(f)

    
    # Save corrected JSON
    with open(os.path.join(corrected_json_path, f"{video_id}.json"), 'w') as f_out:
        json.dump(corrected_data, f_out)

print(f"\n✅ Finished correcting and saving {len(candidates_for_tracking)} videos.")
print(f"📁 Output stored in: {corrected_json_path}")

# Copy over the untouched JSONs (not in candidates_for_tracking)
untouched_videos = set(all_data.keys()) - set(candidates_for_tracking.keys())

for video_id in untouched_videos:
    filepath = os.path.join(base_dir, 'ohp_poses', sample_class, f"{video_id}.json")
    dest_path = os.path.join(corrected_json_path, f"{video_id}.json")
    
    if os.path.exists(filepath):
        with open(filepath, 'r') as f_src, open(dest_path, 'w') as f_dst:
            json.dump(json.load(f_src), f_dst)

print(f"📄 Copied {len(untouched_videos)} untouched JSONs to complete the corrected dataset.")



🔄 Reassigning ID 281 → 284 in video 64307_1 (frame 111)
🔄 Reassigning ID 281 → 284 in video 64307_1 (frame 112)
🔄 Reassigning ID 281 → 284 in video 64307_1 (frame 113)
🔄 Reassigning ID 312 → 315 in video 64482_1 (frame 91)
🔄 Reassigning ID 312 → 315 in video 64482_1 (frame 92)
🔄 Reassigning ID 312 → 315 in video 64482_1 (frame 93)
🔄 Reassigning ID 312 → 315 in video 64482_1 (frame 94)
🔄 Reassigning ID 312 → 315 in video 64482_1 (frame 95)
🔄 Reassigning ID 1391 → 1390 in video 70925_3 (frame 26)
🔄 Reassigning ID 1391 → 1390 in video 70925_3 (frame 27)
🔄 Reassigning ID 1391 → 1390 in video 70925_3 (frame 28)
🔄 Reassigning ID 1390 → 1391 in video 70925_3 (frame 26)
🔄 Reassigning ID 1390 → 1391 in video 70925_3 (frame 27)
🔄 Reassigning ID 1390 → 1391 in video 70925_3 (frame 28)

✅ Finished correcting and saving 82 videos.
📁 Output stored in: ../../data\ohp_poses_corrected\elbows_error
📄 Copied 377 untouched JSONs to complete the corrected dataset.


# Stamatia: take only the "main person" from the JSONs


In [142]:
# # Dont use this if you want all the people in the video.
# # This bbg extracts "the main person" from each video based on the largest bounding box area of keypoints.
# # This "main person" is defined as the person with the largest bounding box in the **first frame** of each video.
# # Should redefine that 


# main_person = None
# counter = 0
# main_person_keypoints = {}

# if not extract_main_person:
#     exit()

# for video in all_data.keys():
#     main_person = None
#     largest_area = 0
#     for frame in all_data.get(video).get('keypoints'):
#         if len(frame.keys())>0:
#             for (person, person_keypoints) in frame.items():  # each is a list of keypoints
#                 bbox = get_bounding_box(person_keypoints, threshold=0.2)  # optional threshold
#                 if bbox:
#                     area = bbox_area(bbox)
#                     if area > largest_area:
#                         largest_area = area
#                         main_person = {
#                             "bbox": bbox,
#                             "keypoints": person_keypoints,
#                             "area": area,
#                             "person_id": person
#                         }
#             counter+=1
#             if main_person:
#                 print("Main person bounding box:", main_person["bbox"], video, counter)
#                 print(main_person['area'])
#                 print(main_person['person_id'])
#             break
#     main_persons_frames = []
#     for frame in all_data.get(video).get('keypoints'):
#         if frame.get(main_person['person_id']):
#             main_persons_frames.append(frame.get(main_person['person_id'])[:17])
        
#     main_person_keypoints[video] = {main_person['person_id']: main_persons_frames}


# From JSON to PKL


In [143]:
import itertools

all_keypoints= {}
for video in all_data.keys():
    persons_frames = {}
    all_keypoints[video] = []
     
    people_lst = list([list(all_data.get(video).get('keypoints')[i].keys()) for i in range(len(all_data.get(video).get('keypoints')))])
    people_set = list(set(itertools.chain.from_iterable(people_lst)))
    for person in people_set:
            persons_frames[person] = []
    
    for frame in all_data.get(video).get('keypoints'):
        for person in frame.keys():
            persons_frames[person].append(frame.get(person)[:17])
            
    all_keypoints[video].append(persons_frames)


In [144]:
# len(all_keypoints['62794_6'][0]['44'])

In [145]:
#main_person_keypoints.get('80756_1').get('1860')[0]

In [146]:
# if extract_main_person:
#     keypoints = main_person_keypoints.keys()
# else:
#     pass

In [147]:
# pip install scikit-learn

In [148]:
import random
from sklearn.model_selection import train_test_split

video_ids = list(all_keypoints.keys())
random.seed(42)

train_val, test = train_test_split(video_ids, test_size=0.10, random_state=42)

val_size = 0.1111
train, val = train_test_split(train_val, test_size=val_size, random_state=42)

split = {
    'train': train,
    'val': val,
    'test': test
}

In [149]:
print(len(train), len(test), len(val))

367 46 46


In [150]:
import numpy as np

coords = {}
confidences = {}

# Iteramos sobre todos los videos en all_keypoints
for video_id in all_keypoints.keys():
    persons_data = all_keypoints[video_id][0]  # dict: person_id -> list of frames

    person_ids = list(persons_data.keys())
    num_persons = len(person_ids)
    num_frames = max(len(persons_data[pid]) for pid in person_ids)
    num_keypoints = len(persons_data[person_ids[0]][0])  # assumed 17 keypoints

    # Inicializamos arrays vacíos
    keypoint_array = np.zeros((num_persons, num_frames, num_keypoints, 2), dtype='float32')
    score_array = np.zeros((num_persons, num_frames, num_keypoints), dtype='float32')


    for m, pid in enumerate(person_ids):
        frames = persons_data[pid]
        for t, frame in enumerate(frames):
            for v, kp in enumerate(frame):
                keypoint_array[m, t, v] = kp[:2]
                score_array[m, t, v] = kp[2] if len(kp) > 2 else 0.0

    coords[video_id] = keypoint_array
    confidences[video_id] = score_array


In [151]:
final_dict = {}
final_dict['split'] = {'train': train, 'test': test, 'val': val}
final_dict['annotations'] = []

for video_id in all_keypoints.keys():
    final_dict['annotations'].append({
        'frame_dir': video_id,
        'total_frames': video_properties[video_id]['frame_count'],
        'img_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'original_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'label': 0,
        'keypoint': coords[video_id],  # shape [M, T, V, C]
        'keypoint_score': confidences[video_id]  # shape [M, T, V]
    })


In [152]:
final_dict['annotations'][0]

{'frame_dir': '62805_6',
 'total_frames': 48,
 'img_shape': (600, 480),
 'original_shape': (600, 480),
 'label': 0,
 'keypoint': array([[[[148.6352  , 170.44833 ],
          [142.7023  , 163.54913 ],
          [145.70682 , 161.94452 ],
          ...,
          [511.09167 , 235.31567 ],
          [570.91364 , 109.866425],
          [603.3789  , 223.05258 ]],
 
         [[149.73306 , 169.0805  ],
          [177.24146 , 105.07037 ],
          [147.01593 , 161.0906  ],
          ...,
          [510.97034 , 235.97339 ],
          [569.745   , 109.49628 ],
          [603.8702  , 222.57098 ]],
 
         [[150.8147  , 165.46503 ],
          [145.33044 , 159.0509  ],
          [148.50717 , 157.03455 ],
          ...,
          [509.3875  , 234.99701 ],
          [569.5072  , 109.370544],
          [603.33276 , 222.36609 ]],
 
         ...,
 
         [[149.0133  , 174.24326 ],
          [139.87799 , 167.23444 ],
          [142.00195 , 164.61475 ],
          ...,
          [518.36304 , 224.8772

In [153]:
import pickle


with open(os.path.join(f'{sample_class}.pkl'), 'wb') as handle:
    pickle.dump(final_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [154]:
from joblib import load

obj = load(f'{sample_class}.pkl')
print(type(obj))

<class 'dict'>


In [155]:
obj.get('split')

{'train': ['67978_4',
  '63309_8',
  '74574_4',
  '71960_2',
  '71001_3',
  '62993_1',
  '72299_5',
  '70957_1',
  '78073_1',
  '66350_4',
  '63995_1',
  '66322_3',
  '77786_1',
  '70400_4',
  '67410_3',
  '73033_5',
  '65348_4',
  '63159_8',
  '70961_2',
  '64198_6',
  '80183_4',
  '64386_3',
  '64336_1',
  '67883_4',
  '64899_1',
  '75014_1',
  '63106_6',
  '74680_3',
  '69966_4',
  '67795_2',
  '63919_3',
  '68364_1',
  '68568_3',
  '68396_6',
  '72575_2',
  '70074_1',
  '70322_2',
  '68859_1',
  '65183_4',
  '67175_3',
  '68335_3',
  '63729_5',
  '80638_4',
  '69055_3',
  '65369_1',
  '68280_1',
  '64673_3',
  '72247_1',
  '66353_3',
  '70674_1',
  '69572_7',
  '69484_3',
  '65857_1',
  '64420_3',
  '70023_2',
  '73152_5',
  '67875_5',
  '64607_2',
  '68385_1',
  '76981_4',
  '73121_3',
  '71164_4',
  '69510_1',
  '74909_8',
  '73040_2',
  '67852_1',
  '63390_7',
  '66815_1',
  '76140_3',
  '64108_11',
  '68656_5',
  '66495_4',
  '63203_3',
  '70995_4',
  '66165_6',
  '67572_1',
  

In [156]:
obj.get('annotations')

[{'frame_dir': '62805_6',
  'total_frames': 48,
  'img_shape': (600, 480),
  'original_shape': (600, 480),
  'label': 0,
  'keypoint': array([[[[148.6352  , 170.44833 ],
           [142.7023  , 163.54913 ],
           [145.70682 , 161.94452 ],
           ...,
           [511.09167 , 235.31567 ],
           [570.91364 , 109.866425],
           [603.3789  , 223.05258 ]],
  
          [[149.73306 , 169.0805  ],
           [177.24146 , 105.07037 ],
           [147.01593 , 161.0906  ],
           ...,
           [510.97034 , 235.97339 ],
           [569.745   , 109.49628 ],
           [603.8702  , 222.57098 ]],
  
          [[150.8147  , 165.46503 ],
           [145.33044 , 159.0509  ],
           [148.50717 , 157.03455 ],
           ...,
           [509.3875  , 234.99701 ],
           [569.5072  , 109.370544],
           [603.33276 , 222.36609 ]],
  
          ...,
  
          [[149.0133  , 174.24326 ],
           [139.87799 , 167.23444 ],
           [142.00195 , 164.61475 ],
           .